In [ ]:
import os
import yaml
import tempfile 

os.environ['TMPDIR'] = '/path/to/your/large/disk/tmp'  
tempfile.tempdir = '/diskpool/tmp'  
from alfworld.agents.environment import get_environment 

In [ ]:
config_file = '/data/home/zhangjs/disk/project/verl-agent/agent_system/environments/env_package/alfworld/configs/config_tw.yaml'

In [ ]:
eval_split = 'train'
batch_size = 3553

In [ ]:
with open(config_file) as reader:
    config = yaml.safe_load(reader) 

env_type = config['env']['type'] 
env_class = get_environment(env_type) 

envs = env_class(config, train_eval=eval_split) 
env = envs.init_env(batch_size=batch_size) 
print() 
class Env:
    def __init__(self,env):
        self.env = env
        self.last_command = None
        self.auto_reset = True

    def step(self,action):
        if self.last_command is not None and self.last_command[2]:
            # This means the game has ended
            obs, reward, done, infos = self.last_command

            if self.auto_reset:
                reward, done = 0., False
                obs, infos = self.env.call_sync("reset") 
        else:
            self.env.call("step",action) 
        
        results = self.env.result()
        obs, rewards, dones, infos = results
        self.last_command = results

        return obs, rewards, dones, infos
    
    def reset(self):
        self.env.call("reset") 
        infos = self.env.result() 
        
        return obs, infos


class WrapParallel:
    def __init__(self,batch_envs):
        self.batch_envs = batch_envs 
        self.envs = batch_envs.batch_env.envs 
        self.reset_all() 
        
    def reset_all(self):
        obs, info = self.batch_envs.reset() 
        self.wraped_env = [Env(env) for env in self.batch_envs.batch_env.envs] 
        self.env_dict = {}
        self.start_poa_dict = {}
        for filename, env, poa in zip(info['extra.gamefile'],self.batch_envs.batch_env.envs,info['admissible_commands']):
            self.env_dict[filename] = Env(env) 
            self.start_poa_dict[filename] = poa
        return obs,info 
        
    def reset(self,index):
        current_env = self.wraped_env[index]
        obs, infos = current_env.reset() 
        return obs, infos

    def step(self,index,action):
        obs, scores, dones, infos = self.wraped_env[index].step(action)
        return obs, scores, dones, infos

    def step_filename(self,filename,action):
        obs, scores, dones, infos = self.env_dict[filename].step(action)
        return obs, scores, dones, infos 
    
    def get_start_poa_filename(self,filename):
        return self.start_poa_dict[filename]


envs = WrapParallel(env)
obs,info = envs.reset_all() 

In [1]:
import json
gt_data = json.load(open('/data/home/zhangjs/disk/project/verl-agent/data_pipelines/datas/vanilla/train_expert_traj.json','r'))

In [2]:
count = 0
for elem in gt_data:
    if elem['short_gt']['win']:
        count += 1

In [3]:
count

3553

In [ ]:
gt_data[0]['game_files'] 

count = 0
gt_alfworld_train = []

for elem in gt_data:
    gamefile = elem['game_files']
    short_gt = elem['short_gt']
    
    if gamefile in set(envs.env_dict.keys()):
        poa_temp = envs.get_start_poa_filename(gamefile)

        count += 1 
        for conv in short_gt['observations']:
            conv['admissible_commands'] = poa_temp
            action = conv['plan']
            observation = conv['observation']
            env_feedback = envs.step_filename(
                filename=gamefile,
                action=action
            )
            admissible_commands = env_feedback[-1]['admissible_commands']
            poa_temp = admissible_commands
    
        gt_alfworld_train.append(elem['short_gt'])

count 